# 04. Text Translation — via LLM Prompting (Azure OpenAI Responses API)

**This is one of two "text translation" notebooks in this folder — read the title carefully.** This notebook
(`04_text_translation.py`) translates text by **prompting a general-purpose chat model** through the Azure
OpenAI Responses API, asking it to detect the source language and produce a tone-preserving translation into a
target language of your choice. The sibling notebook `09_text_translation.py` instead calls the **dedicated
Azure AI Translator REST API** — a purpose-built machine translation service — with no LLM involved. Read both
to see the same task solved two structurally different ways.

**Difficulty:** Beginner


## Prerequisites

**Python packages** (already covered by the repo's root `requirements.txt`):
- `openai`, `azure-ai-projects`, `azure-identity`, `python-dotenv`

**Azure resources required:**
- An Azure AI Foundry project with a chat-capable model deployment

**Environment variables** (add to root `.env`):
```
AZURE_AI_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_MODEL_DEPLOYMENT=<your-deployment-name>
```

Auth is via `DefaultAzureCredential` (`az login`).


## What You'll Learn

- How to prompt an LLM to translate text while preserving tone/register (not just literal word-for-word MT)
- How to ask the model to also report the detected source language in a structured plain-text format
- How to loop the same translation function over multiple target languages
- How LLM-based translation compares to the dedicated Azure AI Translator service (see notebook 09) in terms
  of guarantees, language coverage, and cost/latency trade-offs


### Step 1 — Connect to the Azure AI Foundry project (same setup as notebooks 01–03)

💡 **Exam tip:** For AI-102, the **Azure AI Translator** service (Text Translation) is the certified, purpose-
built machine translation offering — it supports 100+ languages/dialects, has documented latency/throughput
characteristics, supports transliteration and profanity filtering, and can be extended with **Custom
Translator** for domain-specific terminology. An LLM-prompted translation like this notebook's has none of
those formal guarantees, but can outperform literal MT on tone, idiom, and context-awareness for some use
cases.

🔄 **Alternatives:** See notebook 09 for the dedicated-service approach to the exact same translation task.


In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
deployment_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT"]

project = AIProjectClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(),
)

openai = project.get_openai_client()

### Step 2 — Define the source text and a reusable `translate()` function

`translate(text, target_language)` builds a system prompt on the fly for whichever target language is passed
in, instructing the model to:
- translate into `target_language`
- preserve line breaks/formatting
- preserve tone/register (urgent, formal, casual, etc.) rather than a flat literal translation
- respond in a fixed plain-text format: `SOURCE LANGUAGE: ...` then `TRANSLATION: ...`

Note this "fixed format" is requested via prompt instructions only (like notebook 01, not notebook 02) — so,
same caveat as before, it's not schema-enforced.

💡 **Exam tip:** Asking the model to report the detected source language mirrors a real Translator service
feature: when you omit the `from` (source language) parameter in a Translator API call, the service
auto-detects it for you and returns the detected language code alongside the translation — see notebook 09.

🔄 **Alternatives:** Instead of re-prompting per target language in a loop (as this script does), a single
Translator API call can translate to *multiple* target languages at once (notebook 09's `targets` array) —
more efficient when you know all target languages up front.


In [ ]:
source_text = """
Hi team, our VPN keeps dropping every 10 minutes since the last update.
This is affecting our whole sales team and we need this fixed today.
"""

def translate(text: str, target_language: str) -> str:
    system_prompt = f"""
    You are a professional translator. Translate the user's text into
    {target_language}. Preserve the original line breaks and formatting.
    Preserve the tone and register of the original (e.g. formal, urgent,
    casual) rather than producing a flat, literal, word-for-word translation.

    Respond in exactly this plain-text format, with no extra commentary:

    SOURCE LANGUAGE: <detected source language>
    TRANSLATION: <the translated text>
    """

    response = openai.responses.create(
        model=deployment_name,
        input=[
            {"type": "message", "role": "system", "content": system_prompt},
            {"type": "message", "role": "user", "content": text}
        ]
    )
    return response.output_text

### Step 3 — Translate into multiple target languages

This loop calls `translate()` once per target language, making one full LLM round-trip per language. For two
languages that's two calls; for ten target languages, that's ten separate calls — each with its own latency
and token cost.

💡 **Exam tip:** Contrast this fan-out pattern with the dedicated Translator API's `targets` array (notebook
09), where a single REST call can return translations into several target languages at once — an important
cost/latency distinction to know for solution design questions on the exam.

🔄 **Alternatives:** Batch all target languages into one LLM prompt asking for a JSON object keyed by language
code (combine with notebook 02's structured-output technique) to cut this down to a single model call instead
of one per language.


In [ ]:
for target in ["French", "Japanese"]:
    print(f"--- Translating to {target} ---")
    print(translate(source_text, target))
    print()

## Summary

This notebook translated a short support message into French and Japanese by prompting a general-purpose LLM,
one target language per model call, with instructions to preserve tone and report the detected source
language. It's flexible and can be tone-aware, but has no formal language-coverage guarantee, no confidence
scoring, and no cost/latency advantage of a purpose-built batch translation call. See notebook 09 for the same
task done with the dedicated Azure AI Translator service.


## Try It Yourself

1. **Easy:** Add two more target languages to the loop (e.g. `"German"`, `"Arabic"`) and observe how well tone
   preservation holds up across very different language families.
2. **Intermediate:** Parse the `SOURCE LANGUAGE:` / `TRANSLATION:` plain-text format into a Python dict instead
   of just printing it — handle the case where the model doesn't follow the format exactly.
3. **Advanced:** Run the same `source_text` through both this notebook and notebook 09, and compare: which
   one better preserves the urgency of "we need this fixed today"? Which one is faster / would scale better to
   translating thousands of tickets per day?
